# Module 4 — Q&A RAG Pipeline

**What RAG is, in one paragraph:** instead of asking an LLM to answer purely from what it "remembers," you first **retrieve** the most relevant pieces of text from your own knowledge base (here: real past support Q&A pairs), then **paste those pieces into the prompt** as context, and ask the LLM to answer **using only that context**. This grounds the answer in real data and lets you cite/verify where the answer came from.

**The 3 pieces you're building:**
1. **Embeddings** — turn every past support question into a vector (a list of numbers) that captures its meaning. We use `sentence-transformers/all-MiniLM-L6-v2`.
2. **Vector store** — a searchable index of those vectors, so given a *new* customer question we can find the most similar past questions instantly. We use **FAISS** (local, free, no account needed — the assignment allows this as an alternative to cloud Qdrant).
3. **Generation** — an LLM (via the free **Groq** API, model `openai/gpt-oss-20b`) that reads the retrieved Q&A pairs and writes a final answer in natural language.

**Where to run this:** any Jupyter environment, CPU is fine. You will need a **free Groq API key** — see Step 0 below.

## Step 0 — Get a free Groq API key (one-time, ~2 minutes)
1. Go to https://console.groq.com and sign up (free).
2. Go to **API Keys** in the left sidebar → **Create API Key**.
3. Copy the key (starts with `gsk_...`).
4. Paste it below where it says `GROQ_API_KEY = "..."` — or, better, set it as an environment variable so you never hardcode it in the notebook:
   - In Colab: `import os; os.environ["GROQ_API_KEY"] = "gsk_..."`
   - Locally: `export GROQ_API_KEY=gsk_...` in your terminal before launching Jupyter.

In [12]:
# 1. Install dependencies
!pip install -q sentence-transformers faiss-cpu groq datasets pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import os

# Set GROQ_API_KEY in the environment before starting Jupyter.
# Example (PowerShell): $env:GROQ_API_KEY = "gsk_..."
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("API key loaded:", bool(GROQ_API_KEY))

API key loaded: True


In [23]:
import os

print("API key loaded:", bool(os.getenv("GROQ_API_KEY")))

API key loaded: True


In [24]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [14]:
# 3. Load the knowledge base: the same Bitext dataset, using instruction+response as Q&A chunks
from datasets import load_dataset
import pandas as pd

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
kb_df = ds["train"].to_pandas()[["instruction", "response", "intent", "category"]]
print(kb_df.shape)
kb_df.head()

(26872, 4)


,instruction,response,intent,category
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


In [15]:
# 4. Build embeddings for every past customer question ("instruction")
# We embed the QUESTIONS (so a new question can be matched to similar past questions),
# and keep the paired RESPONSE as the grounding text we hand to the LLM.
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

questions = kb_df["instruction"].tolist()
embeddings = embedder.encode(
    questions,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalize so we can use inner product as cosine similarity
)
embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/210 [00:00<?, ?it/s]

(26872, 384)

In [16]:
# 5. Build a FAISS index (a fast nearest-neighbor search structure over the vectors)
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # inner product == cosine similarity, since vectors are normalized
index.add(embeddings)
print("Vectors in index:", index.ntotal)

Vectors in index: 26872


In [17]:
# 6. Save the index + the metadata (questions/responses) needed to look up hits later
import os, pickle

os.makedirs("../models/rag_store", exist_ok=True)
faiss.write_index(index, "../models/rag_store/kb.index")
kb_df.to_pickle("../models/rag_store/kb_metadata.pkl")
print("Saved FAISS index and metadata to ../models/rag_store/")

Saved FAISS index and metadata to ../models/rag_store/


In [18]:
# 7. Retrieval function: given a new customer question, find the top-k most similar past Q&A pairs
def retrieve(query: str, k: int = 3):
    query_vec = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, idxs = index.search(query_vec, k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        row = kb_df.iloc[idx]
        hits.append({
            "score": float(score),
            "matched_question": row["instruction"],
            "response": row["response"],
        })
    return hits

# quick test
for h in retrieve("where is my order, it's been 5 days"):
    print(round(h["score"], 3), "|", h["matched_question"])

0.668 | where could i see when my order is gonna arrive
0.665 | where to see how long it takes for my order to arrive?
0.661 | where to see when my order is gonna arrive


In [26]:
# 8. Generation step: call Groq's LLM with the retrieved context, following the assignment's prompt template
from groq import AuthenticationError, Groq

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
GROQ_MODEL = "openai/gpt-oss-20b"   # or "openai/gpt-oss-120b" for higher quality / slower+costlier

def build_prompt(user_message: str, retrieved_chunks: list, detected_sentiment: str = "neutral") -> list:
    context_block = "\n\n".join(
        f"Q: {c['matched_question']}\nA: {c['response']}" for c in retrieved_chunks
    )
    system_prompt = (
        "You are a helpful, professional customer support assistant for an online retailer. "
        "Answer the customer's question using ONLY the information in the retrieved support "
        f"responses below. If the customer sounds frustrated ({detected_sentiment}), acknowledge "
        "that before answering. If the retrieved context does not cover the question, say so "
        "honestly and offer to escalate to a human agent rather than guessing."
    )
    user_prompt = (
        f"Context (retrieved past support responses):\n{context_block}\n\n"
        f'Customer question: "{user_message}"'
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

def generate_answer(user_message: str, detected_sentiment: str = "neutral", k: int = 3) -> str:
    hits = retrieve(user_message, k=k)
    if groq_client is None:
        return "Groq API key not configured. Set GROQ_API_KEY and rerun the generation cells."

    messages = build_prompt(user_message, hits, detected_sentiment)
    try:
        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            temperature=0.3,
            max_tokens=300,
        )
    except AuthenticationError:
        return "Groq rejected the API key. Set a valid GROQ_API_KEY and rerun the generation cells."
    return completion.choices[0].message.content

In [27]:
# 9. End-to-end test
print(generate_answer(
    "I ordered a jacket 8 days ago and it still says 'processing', I'm getting worried",
    detected_sentiment="negative",
))

I’m sorry to hear you’re feeling worried about the delay. To help you get the most accurate update on your jacket, could you please share your Order Number or Tracking Number? Once we have that, we can check the current status and give you an estimated delivery date. Thank you for your patience.
